In [27]:
import polars as pl

In [28]:
rucs_sri = pl.read_parquet(r"..\bases_datos\base_rucs_sri.parquet")
catastro = pl.read_parquet(r"..\bases_datos\catastro.parquet")
excel = pl.read_excel(r"..\bases_datos\ciiu_nivel6.xlsx")

In [29]:
#Procesamiento Catastro
catastro = catastro.rename({'actividad_economica': 'actividad_economica_catastro'}).unique(pl.col("numero_ruc"))

In [30]:
#Procesamiento del excel

#Eliminamos Columnas innecesarias
excel = excel.drop(["NIVEL", "__UNNAMED__0"])

#Estandarizamos el formato para que coincida con los demás df
excel = (excel
         .with_columns(pl.col("CODIGO").str.replace(r"\.",""))
         .with_columns(pl.col("DESCRIPCION").str.strip_chars().str.to_uppercase())
         )

excel = excel.rename({"DESCRIPCION": "DESCRIPCION_INEC"})

In [31]:
#Comprobamos si hay más ciiu que en alguna u otra base

longitud_ciiu_catastros_no_inec = len(set(catastro['codigo_ciiu'])-set(excel['CODIGO']))
if longitud_ciiu_catastros_no_inec> 0:
    print(f"Hay {longitud_ciiu_catastros_no_inec} más códigos en catastro que en en inec")
else:
    print("Hay igual o más en Inec que en catastro") #6704268

Hay 157 más códigos en catastro que en en inec


In [32]:
#Procesamiento Base_rucs_sri

#Nos quedamos rucs_validos
rucs_sri = rucs_sri.filter(
    pl.col("numero_ruc").cast(pl.Utf8).str.len_chars() >= 9
)

#Quitamos duplicados por fecha
rucs_sri = (rucs_sri
            .sort("fecha_actualizacion", descending = True)
            .unique(subset = ["numero_ruc"]))


#Nos quedamos con las columnas importantes
rucs_sri = rucs_sri.select(["numero_ruc","actividad_economica","razon_social", "estado_contribuyente" , "clase_contribuyente", "categoria", "obligado_llevar_contabilidad", "agente_retencion", "contribuyente_especial"])

In [33]:
# Obtener agregamos las columnas rucs_sri <--- catastro <--- Inec
catastro_inec  = catastro.join(excel, right_on = "CODIGO", left_on = "codigo_ciiu", how = "left")
set_ciiu_catastro_no_inec = set(catastro_inec.filter(pl.col("DESCRIPCION_INEC").is_null())['codigo_ciiu'])

sri_catastro_inec = rucs_sri.join(catastro_inec, on = "numero_ruc", how= "left" )

In [34]:
#Filtramos los que tienen actividad economica en sri pero no tienen INEC

catastro_sri_no_inec = sri_catastro_inec.filter(
    pl.col("DESCRIPCION_INEC").is_null() & pl.col("actividad_economica").is_not_null())

lognitud_catastro__sri_no_inec = len(catastro_sri_no_inec.unique(pl.col("actividad_economica")))
print(f"Hay {lognitud_catastro__sri_no_inec} actividades_economicas en tienen catastro y sri pero no inec")

Hay 1333 actividades_economicas en tienen catastro y sri pero no inec


In [35]:
#Trabajamos sobre los que no tienen inec
catastro_sri_no_inec = catastro_sri_no_inec.select(["numero_ruc","actividad_economica","razon_social", "estado_contribuyente" , "clase_contribuyente", "categoria", "obligado_llevar_contabilidad", "agente_retencion", "contribuyente_especial"])
catastro_sri_no_inec_aumentado_inec = catastro_sri_no_inec.join(excel, left_on = "actividad_economica", right_on = "DESCRIPCION_INEC", how = "left")

In [36]:
#Vemos los que faltan para la corrección semántica
faltantes_correccion_semantica = catastro_sri_no_inec_aumentado_inec.filter(pl.col("actividad_economica").is_not_null() & pl.col("CODIGO").is_null())

In [37]:
faltantes_correccion_semantica = faltantes_correccion_semantica.with_columns(pl.col("actividad_economica").str.strip_chars())

In [38]:
#Lectura de las correcciones que faltan
correciones = pl.read_excel(r"..\bases_datos\correccion_final.xlsx").select(["actividad_economica","codigo_ciiu","descripcion_ciiu"]).with_columns(pl.col("descripcion_ciiu").str.to_uppercase())

In [39]:
corregido_semanticamente = faltantes_correccion_semantica.join(correciones, on = "actividad_economica", how = "left").drop(pl.col("CODIGO")).rename({"codigo_ciiu": "CODIGO", "descripcion_ciiu": "DESCRIPCION_INEC"})

In [40]:
#Vamos a juntar todo:

#Tenemos los que fueron corregidos al hacer join entre catastro e inec'. Este esta desagregado hasta RUCS
corregidos_catastro_inec = sri_catastro_inec.filter(
    pl.col("DESCRIPCION_INEC").is_not_null()
    )

#Tenemos los que fueron corregidos a partir de los que tienen sri y no inec. Este está desagregado hasta RUCS
corregidos_sri_no_inec = catastro_sri_no_inec_aumentado_inec.filter(pl.col("actividad_economica").is_not_null() & pl.col("CODIGO").is_not_null())

#Tenemos las correcciones de los que no tenían match directo y se hizo semanticamente.



In [41]:
corregidos_catastro_inec = (
    corregidos_catastro_inec
    .drop("actividad_economica_catastro")
    .rename({"codigo_ciiu": "CODIGO"})
    .with_columns(
        pl.lit(1).alias("correcion_catastro_inec_ciiu")
    )
)

corregidos_sri_no_inec = (
    corregidos_sri_no_inec
    .with_columns([
        pl.col("razon_social").alias("DESCRIPCION_INEC"),
        pl.lit(1).alias("correcion_sri_inec_desc")
    ])
)

corregido_semanticamente = (
    corregido_semanticamente
    .with_columns(
        pl.lit(1).alias("correcion_semantica")
    )
)

base_rucs_sri_corregido_catastro = pl.concat([corregidos_catastro_inec, corregidos_sri_no_inec, corregido_semanticamente], how = 'diagonal')

In [48]:
rucs_sri

numero_ruc,actividad_economica,razon_social,estado_contribuyente,clase_contribuyente,categoria,obligado_llevar_contabilidad,agente_retencion,contribuyente_especial
f64,str,str,str,str,str,i64,i64,i64
1.3165e12,"""ACTIVIDADES DE SERVICIOS DIVER…","""ROBALINO CHOEZ FELIX FRANSESCO""","""ACTIVO""","""RIMPE""","""NEGOCIO POPULAR""",0,0,0
6.0234e11,"""VENTA AL POR MENOR DE FRUTAS, …","""SHAGÑAY GADÑAY ZOILA MARINA""","""SUSPENDIDO""","""RIMPE""","""NEGOCIO POPULAR""",0,0,0
7.0658e11,"""ACTIVIDADES DE CONSULTORÍA DE …","""CARRION ROMERO DUNIA ANABEL""","""ACTIVO""","""GENERAL""","""""",0,0,0
9.1651e11,"""MANTENIMIENTO Y REPARACIÓN DE …","""BENAVIDES OSTAIZA MARIO GIOVAN…","""SUSPENDIDO""","""GENERAL""","""""",0,0,0
9.0913e11,"""VENTA AL POR MENOR DE PESCADO,…","""TIGRERO GONZALEZ PEDRO HIGINIO""","""SUSPENDIDO""","""GENERAL""","""""",0,0,0
…,…,…,…,…,…,…,…,…
1.0625e11,"""VENTA AL POR MAYOR DE PRENDAS …","""IDROVO IDROVO MARIA CRISTINA""","""SUSPENDIDO""","""GENERAL""","""""",0,0,0
2.0151e11,"""CRÍA Y REPRODUCCIÓN DE GANADO …","""COLLAY TOALOMBO LUIS RAMIRO""","""SUSPENDIDO""","""GENERAL""","""""",0,0,0
9.9057e11,"""RESTAURANTES, CEVICHERÍAS, PIC…","""FUENTES RUIZ VICTOR FLORENCIO""","""PASIVO""","""GENERAL""","""""",0,0,0


In [ ]:
base_rucs_sri_corregido_catastro.unique("numero_ruc").write_parquet(r"..\bases_datos\base_rucs_corregida.parquet")